# ADS 542 Statistical Learning Final Project

## Bank Term Deposit Subscription Prediction

This notebook builds a binary classification model for the Bank Marketing dataset. The goal is to predict whether a bank client subscribes to a term deposit. The target column is `y`, where `yes` means the client subscribed and `no` means the client did not subscribe.

The notebook follows the same general structure as the Streamlit app and prepares a final pipeline that can be saved as `rf_pipeline.sav`.


## 1. Import Libraries

First, I import the main libraries for data loading, preprocessing, modeling, evaluation, and saving the final pipeline.


In [ ]:
from pathlib import Path
import pickle
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


## 2. Load the Dataset

The notebook is designed to run from inside the `ADA-PROJECT-FINAL` folder. The dataset path is relative to that folder.


In [ ]:
data_path = Path("bank-additional") / "bank-additional.csv"
df = pd.read_csv(data_path, sep=";")
df.head()


## 3. Basic Dataset Information

Here I check the dataset size, preview the first rows, list the columns, check missing values, and review the target distribution.


In [ ]:
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nMissing values by column:")
print(df.isna().sum())

print("\nTarget class distribution:")
print(df["y"].value_counts())
print("\nTarget class distribution (%):")
print(df["y"].value_counts(normalize=True).round(3))


## 4. Data Cleaning Checks

Before modeling, I check for duplicate rows and missing values. The dataset is mostly clean because it does not contain regular missing values. Some categorical values may be stored as `unknown`, but those are treated as valid categories in this project.


In [ ]:
duplicate_count = df.duplicated().sum()
missing_count = df.isna().sum().sum()

print("Duplicate rows:", duplicate_count)
print("Total missing values:", missing_count)

if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicates were removed. New shape:", df.shape)
else:
    print("No duplicate rows were found.")

if missing_count == 0:
    print("The dataset has no regular missing values, so no imputation is needed.")


## 5. Feature Engineering

The Streamlit app uses several engineered features. I recreate those features here so the training pipeline uses the same type of inputs as the app. I also remove `duration` because it is only known after a phone call ends, so using it would cause data leakage.


In [ ]:
def add_engineered_features(data):
    data = data.copy()

    data["has_university_degree"] = (data["education"] == "university.degree").astype(int)
    data["is_high_campaign"] = (data["campaign"] > 3).astype(int)

    data["white_collar"] = data["job"].isin(["admin.", "technician", "management"]).astype(int)
    data["blue_collar"] = data["job"].isin(["blue-collar", "services", "housemaid"]).astype(int)
    data["other_collar"] = data["job"].isin([
        "retired", "self-employed", "entrepreneur", "unemployed", "student"
    ]).astype(int)

    data["econ_interact"] = data["euribor3m"] * data["emp.var.rate"]
    data["age_euribor_interact"] = data["age"] * data["euribor3m"]
    data["campaign_conf_interact"] = data["campaign"] * data["cons.conf.idx"]
    data["cpi_euribor_diff"] = data["cons.price.idx"] - data["euribor3m"]

    data["previous_contact"] = (data["pdays"] != 999).astype(int)
    data["has_multiple_loans"] = ((data["housing"] == "yes") & (data["loan"] == "yes")).astype(int)
    data["economic_stress"] = (data["emp.var.rate"] < 0).astype(int)

    if "marital" in data.columns:
        data["married"] = (data["marital"] == "married").astype(int)

    return data

model_df = add_engineered_features(df)

if "duration" in model_df.columns:
    model_df = model_df.drop(columns=["duration"])

model_df.head()


## 6. Split X and y

The target `y` is converted from `yes` and `no` into 1 and 0. I also keep the features compatible with the Streamlit app, so columns not collected by the app are not used in the final model.


In [ ]:
y = model_df["y"].map({"yes": 1, "no": 0})
X = model_df.drop(columns=["y"])

# These columns are not entered in the Streamlit app, so they are removed from the training inputs.
app_unused_columns = ["default", "day_of_week", "marital"]
X = X.drop(columns=[col for col in app_unused_columns if col in X.columns])

print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts())


## 7. Train/Test Split

I use a stratified train/test split so the class balance stays similar in both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Training target distribution:")
print(y_train.value_counts(normalize=True).round(3))
print("Test target distribution:")
print(y_test.value_counts(normalize=True).round(3))


## 8. Preprocessing

Numerical columns are scaled with `StandardScaler`. Categorical columns are converted with `OneHotEncoder`. This lets the models use both numeric and text-based features correctly.


In [ ]:
numerical_features = X_train.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ]
)


## 9. Feature Selection

I use `SelectKBest` with mutual information. This is a simple feature selection method that keeps the most useful encoded features for predicting the target.


In [ ]:
selector = SelectKBest(score_func=mutual_info_classif, k=25)


## 10. Train and Compare Models

I compare Logistic Regression, Random Forest, and Gradient Boosting. Because the target classes are imbalanced, I include SMOTE inside the pipeline after preprocessing and feature selection.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

results = []
fitted_pipelines = {}

for name, model in models.items():
    pipeline = ImbPipeline(steps=[
        ("preprocessing", preprocessor),
        ("feature_selection", SelectKBest(score_func=mutual_info_classif, k=25)),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", model),
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline.named_steps["model"], "predict_proba"):
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        roc_auc = np.nan

    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc,
    })

    fitted_pipelines[name] = pipeline

results_df = pd.DataFrame(results).sort_values(by="f1", ascending=False)
results_df


## 11. Detailed Metrics for the Best Initial Model

The table gives a quick comparison. Here I print the confusion matrix and classification report for the best initial model based on F1-score.


In [ ]:
best_initial_name = results_df.iloc[0]["model"]
best_initial_pipeline = fitted_pipelines[best_initial_name]
y_pred_best = best_initial_pipeline.predict(X_test)

print("Best initial model:", best_initial_name)
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_best))
print("\nClassification report:")
print(classification_report(y_test, y_pred_best))


## 12. Hyperparameter Tuning

Random Forest is a strong and easy-to-explain model for this project, so I tune it with `GridSearchCV`. The grid is kept small so the notebook can run in a reasonable amount of time.


In [ ]:
rf_pipeline = ImbPipeline(steps=[
    ("preprocessing", preprocessor),
    ("feature_selection", SelectKBest(score_func=mutual_info_classif, k=25)),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])

param_grid = {
    "feature_selection__k": [20, 25, 30],
    "model__n_estimators": [150, 250],
    "model__max_depth": [None, 8, 12],
    "model__min_samples_split": [2, 5],
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)
print("Best cross-validation F1:", round(grid_search.best_score_, 4))


## 13. Final Model Evaluation

After tuning, I evaluate the final Random Forest pipeline on the test set using accuracy, precision, recall, F1-score, ROC-AUC, confusion matrix, and classification report.


In [ ]:
final_pipeline = grid_search.best_estimator_
y_pred_final = final_pipeline.predict(X_test)
y_proba_final = final_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred_final), 4))
print("Precision:", round(precision_score(y_test, y_pred_final), 4))
print("Recall:", round(recall_score(y_test, y_pred_final), 4))
print("F1-score:", round(f1_score(y_test, y_pred_final), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba_final), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_final))

print("\nClassification report:")
print(classification_report(y_test, y_pred_final))


## 14. Save the Final Pipeline

The final pipeline includes preprocessing, feature selection, SMOTE, and the tuned Random Forest model. It is saved as `rf_pipeline.sav`, which is the file used by the Streamlit app.


In [ ]:
model_path = Path("rf_pipeline.sav")

with open(model_path, "wb") as f:
    pickle.dump(final_pipeline, f)

print(f"Final pipeline saved to: {model_path.resolve()}")


## 15. Conclusion

This notebook prepared the bank marketing dataset, removed the leakage column, added app-style engineered features, compared three classification models, tuned a Random Forest model, and saved the final pipeline for use in the Streamlit app.
